In [6]:
# Library imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter
from pathlib import Path
from collections import Counter
from collections import defaultdict
import os

# Visualization settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# mostra o dataframe para caber certinho na tela
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)

In [7]:
# acessar a pasta experiments
dir_experiments="../experiments/prototypeEvaluation/results"

# Estrutura para armazenar os caminhos dos arquivos 
results_files = defaultdict(dict)

for folder in sorted(os.listdir(dir_experiments)):
    path = os.path.join(dir_experiments, folder)
    for file in sorted(os.listdir(path)):
       if file.endswith("_results.csv") or file.endswith("_layer3_test1-classified-images.cvs"):
                results_files[folder][file] = os.path.join(path, file)
           

In [8]:
results_data = []

for folder, contents in results_files.items():
    superpixel = int(folder.split("_")[0].replace('super', ''))
    nimage = int(folder.split("_")[1].replace('images', ''))
    technique = folder.split('_')[2]
    # print(f"{folder=}")
    # print(f"{superpixel=}, {nimage=}, {technique=}")
    for file, dir in contents.items():
        if isinstance(file, str) and file.endswith('_results.csv'):
            results_file = dir
            seed_value = int(file.replace('_results.csv', '').replace('seed', ''))
            with open(results_file, 'r') as f:
                lines = f.readlines()
                if len(lines) >= 4:
                    # Extract accuracy and kappa metrics from the first two lines
                    class1_accuracy, class2_accuracy = map(float, lines[0].strip().split(';')[:2])
                    kappa, global_accuracy = map(float, lines[1].strip().split(';')[:2])
                    # Try to extract nfeat from line 3 or 5 if available
                    if len(lines) > 5 and ': ' in lines[5]:
                        try:
                            nfeat = int(lines[5].strip().split(': ')[1])
                        except (IndexError, ValueError):
                            nfeat = 0
                    elif len(lines) > 3 and ': ' in lines[3]:
                        try:
                            nfeat = int(lines[3].strip().split(': ')[1])
                        except (IndexError, ValueError):
                            nfeat = 0
                    # Append the extracted metrics to the results list
                    results_data.append({
                        'superpixel': superpixel,
                        'nimage': nimage,
                        'technique': technique,
                        'seed': seed_value,
                        'class1_accuracy': class1_accuracy,
                        'class2_accuracy': class2_accuracy,
                        'kappa': kappa,
                        'global_accuracy': global_accuracy,
                        'nfeat': nfeat
                    })


# Convert the results list to a DataFrame
df_technique = pd.DataFrame(results_data)

# Save the DataFrame to a CSV file for further analysis
df_technique.to_csv('prototype_results_summary.csv', index=False)

In [9]:
df_technique.head(100)

,superpixel,nimage,technique,seed,class1_accuracy,class2_accuracy,kappa,global_accuracy,nfeat
0,50,2,cossine,1011,0.874439,0.988266,0.879559,0.973819,24200
1,50,2,cossine,1213,0.887892,0.987614,0.885689,0.974957,24200
2,50,2,cossine,123,0.878924,0.988918,0.884796,0.974957,24200
3,50,2,cossine,2735,0.852018,0.986962,0.860413,0.969835,24200
4,50,2,cossine,42,0.860987,0.988266,0.870948,0.972112,24200
...,...,...,...,...,...,...,...,...,...
75,50,5,euclidean,456,0.892377,0.988918,0.893276,0.976665,60500
76,50,5,euclidean,6854,0.874439,0.991525,0.891591,0.976665,60500
77,50,5,euclidean,7580,0.919282,0.988266,0.907548,0.979511,60500
78,50,5,euclidean,789,0.892377,0.988918,0.893276,0.976665,60500


In [10]:
# Define the metrics to be analyzed (excluding 'nfeat' for summary statistics)
metrics = ['class1_accuracy', 'class2_accuracy', 'kappa', 'global_accuracy', 'nfeat']

# Agrupa por superpixel, nimage e technique e calcula média e desvio padrão para cada métrica
summary_stats = df_technique.groupby(['superpixel', 'nimage', 'technique'])[metrics[:-1]].agg(['mean', 'std']).reset_index()

# Rename columns for easier access (e.g., 'kappa_mean', 'global_accuracy_mean')
summary_stats.columns = ['_'.join(col).strip() for col in summary_stats.columns.values]

# Arrange the DataFrame according to the order of superpixel values
# superpixels_values = sorted(superpixels_values)
# summary_stats = summary_stats.set_index('superpixel_').loc[superpixels_values].reset_index()

# Select only the metrics of interest for visualization and highlight the highest values
summary_stats = summary_stats[['superpixel_', 'nimage_', 'technique_', 'kappa_mean', 'kappa_std', 'global_accuracy_mean', 'global_accuracy_std']].style.highlight_max(
    subset=['kappa_mean', 'global_accuracy_mean'], color='gray'
)

# Format values as percentages for better presentation
summary_stats = summary_stats.format({'kappa_mean': '{:.4%}', 'global_accuracy_mean': '{:.4%}'})

# Display the styled DataFrame
summary_stats

,superpixel_,nimage_,technique_,kappa_mean,kappa_std,global_accuracy_mean,global_accuracy_std
0,50,2,cossine,87.2700%,0.009784,97.2339%,0.002133
1,50,2,euclidean,87.1760%,0.011058,97.2111%,0.002429
2,50,3,cossine,88.4872%,0.014398,97.4900%,0.003157
3,50,3,euclidean,88.4872%,0.014398,97.4900%,0.003157
4,50,4,cossine,89.3676%,0.008361,97.6835%,0.001740
5,50,4,euclidean,89.3676%,0.008361,97.6835%,0.001740
6,50,5,cossine,89.5245%,0.011601,97.7120%,0.002471
7,50,5,euclidean,89.5245%,0.011601,97.7120%,0.002471
